# Presentable Visualizations
- I gotta download some graphs in a cleanlier way to put them on the presentation
- Change titles, background colors, maybe export as one file, etc

In [2]:
import glob
import os
import pandas as pd
import altair as alt
import statsmodels.formula.api as smf

os.makedirs("charts", exist_ok=True)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [11]:
dfs = []
whale_dfs = []
all_charts = []

line_color = "#214c3c" #33-76-60
whale_color = "#D05353" #idk
bg_color = "#fafaf8" #250-250-248

def clean_title(csv_path:str) -> str:
    name = os.path.basename(csv_path).replace("_trades.csv","")
    name = name.replace("_"," ").replace("-"," ")
    return name.title()

def whale_hunting(csv_path: str):
    market_test = pd.read_csv(csv_path)
    market_test = market_test.drop(
        columns=["name", "pseudonym", "bio", "profileImage", "profileImageOptimized", "transactionHash"],
        errors="ignore"
    )

    market_volume = market_test["size"].sum()
    WHALESHARE = 0.05
    whale_threshold = WHALESHARE * market_volume

    df = market_test.copy()

    df["price_outcome"] = df["price"]
    df.loc[df["outcomeIndex"] == 0, "price_outcome"] = (
        1 - df.loc[df["outcomeIndex"] == 0, "price"]
    )

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
    df = df.dropna(subset=["timestamp", "price_outcome"]).sort_values("timestamp").copy()

    df["is_whale"] = df["size"] >= whale_threshold

    title = df.loc[0, "title"]

    # Plot graph
    market_graph = alt.Chart(df).mark_line(color=line_color).encode(
        alt.X("timestamp:T").title("Time"),
        alt.Y("price_outcome:Q").title("Price (Outcome Space)")
    ).properties(
        width=600,
        height=300,
    )
    whale_points = alt.Chart(df[df["is_whale"]]).mark_point(
        filled=True, size=60, color=whale_color
    ).encode(
        alt.X("timestamp:T").title("Time"),
        alt.Y("price_outcome:Q").title("Price (Outcome Space)"),
        tooltip=[
            alt.Tooltip("proxyWallet:N", title="Wallet ID"),
            alt.Tooltip("outcomeIndex:Q", title="Outcome Index"),
            alt.Tooltip("side:N", title="Action"),
            alt.Tooltip("size:Q", title="Trade Size")
        ]
    )
    chart = (market_graph + whale_points).properties(
        width=600,
        height=300,
        title=alt.TitleParams(font="serif", text=title)
    )

    chart.save(f"charts/{title}.png")

    return chart, df[df["is_whale"]].copy()


# do this for trump-28_trades.csv, will-2_trades.csv, will-575_trades.csv
for csv_path in sorted(glob.glob("presenting_csvs/*_trades.csv")):
    chart, whales = whale_hunting(csv_path)
    if chart is not None:
        all_charts.append(chart)
        display(chart)

combined = alt.vconcat(*all_charts, spacing=20).configure_view(stroke=None)
combined.save("charts/pres.png")
combined


alt.LayerChart(...)

alt.LayerChart(...)

alt.VConcatChart(...)